# 🚀 Google Colab Setup Instructions

**IMPORTANT: Enable GPU Runtime**
1. Go to `Runtime` → `Change runtime type`
2. Select `GPU` as Hardware accelerator
3. Choose T4, V100, or A100 (if available)
4. Click `Save`

**Note:** This notebook requires GPU for efficient training. Without GPU, training will be extremely slow.

## 📁 Optional: Mount Google Drive (for data storage)

In [ ]:
# Uncomment the lines below if you want to mount Google Drive
# This is useful for accessing datasets or saving models to Drive

from google.colab import drive
drive.mount('/content/drive')
print("✓ Google Drive mounted at /content/drive")

# QWEN Fine-Tuning for 5G Network Troubleshooting

This notebook fine-tunes QWEN 1.5B models on telecom troubleshooting data using LoRA for efficient training on Google Colab with GPU support.

## Dataset Overview
- **Size**: 112,806 training samples
- **Format**: CSV with ID, question, answer columns
- **Task**: Multi-choice classification (8 root cause options)

- Adjust data paths as needed for your Colab environment

## Requirements- Google Colab with GPU runtime (T4, V100, or A100 recommended)

## 📝 Data Format: Q&A Structure

**Key Design:**
- **Original Question**: Preserved from training data (network data tables + instructions)
- **Engineered Features**: Computed RCA indicators stored in separate column
- **Enhanced Question**: Original question + engineered features combined
- **Answer**: Simple label format (C1-C8)

**DataFrame Columns:**
1. `ID` - Sample identifier
2. `original_question` - Original question from CSV
3. `features_text` - Formatted engineered features (human-readable)
4. `enhanced_question` - Combined question + features (model input)
5. `answer` - Root cause label (model target)
6. `features_dict` - Raw feature values (for analysis)

This structure allows:
- ✅ Easy inspection of features vs questions
- ✅ Flexible training (use original or enhanced questions)
- ✅ Simple Q&A format (no complex chat templates)
- ✅ Better feature engineering tracking


In [1]:
# ============================================================================
# 1. ENVIRONMENT SETUP & IMPORTS
# ============================================================================

import torch
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Check PyTorch and device setup
print("=" * 60)
print("ENVIRONMENT CHECK")
print("=" * 60)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    device = "cuda"
else:
    print("WARNING: GPU not available, using CPU (training will be very slow)")
    device = "cpu"


print(f"Using device: {device}")

ENVIRONMENT CHECK
PyTorch version: 2.9.1
CUDA available: False
Using device: cpu


In [2]:
# Install required packages for Colab
import subprocess
import sys

# packages = [
#     "transformers>=4.36.0",
#     "peft>=0.7.0",  # For LoRA
#     "datasets>=2.14.0",
#     "accelerate>=0.24.0",
#     "bitsandbytes>=0.41.0",  # For 4-bit/8-bit quantization on GPU
#     "scipy",
# ]

# print("Installing required packages...")
# for package in packages:
#     print(f"  Installing {package}...")
#     subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

# print("\n✓ All packages installed")

# Import after installation
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from datasets import Dataset
import gc

print("✓ All imports successful")

# Clear GPU cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("✓ GPU cache cleared")


✓ All imports successful


## 2. Data Loading & Preprocessing

In [3]:
# TRAIN_FILE = "/content/drive/MyDrive/git/the-ai-telco-troubleshooting-challenge/data/train.csv"
# TEST_FILE = "/content/drive/MyDrive/git/the-ai-telco-troubleshooting-challenge/data/phase_1_test.csv"

TRAIN_FILE = "data/train.csv"
TEST_FILE = "data/phase_1_test.csv"


df_train = pd.read_csv(TRAIN_FILE)
display(df_train.head())
print(f"✓ Loaded {len(df_train)} training samples")
print(f"Columns: {df_train.columns.tolist()}")
print(f"\nFirst sample:")
print(f"Question preview: {df_train['question'].iloc[0][:200]}...")
print(f"Answer: {df_train['answer'].iloc[0]}")
print(f"\nAnswer distribution:")
print(df_train['answer'].value_counts())

,ID,question,answer
0,ID_1P7PJMPV0R,Analyze the 5G wireless network drive-test use...,C2
1,ID_8B1D1TUTFA,Analyze the 5G wireless network drive-test use...,C1
2,ID_IGGXMA9GZH,Analyze the 5G wireless network drive-test use...,C2
3,ID_D6C9N2X295,Analyze the 5G wireless network drive-test use...,C2
4,ID_8JC15PNP3Q,Analyze the 5G wireless network drive-test use...,C5


✓ Loaded 2400 training samples
Columns: ['ID', 'question', 'answer']

First sample:
Question preview: Analyze the 5G wireless network drive-test user plane data and engineering parameters.
Identify the reason for the throughput dropping below 600Mbps in certain road sections.
From the following 8 pote...
Answer: C2

Answer distribution:
answer
C5    352
C7    349
C3    330
C2    320
C4    283
C8    277
C1    264
C6    225
Name: count, dtype: int64


## 📊 Enhanced Preprocessing with Type Safety & Feature Engineering

**Key Improvements:**
1. ✅ **Type casting** - Ensures all numeric fields are properly typed (not strings)
2. ✅ **Missing neighbor RSRP fields** - Now included in float casting
3. ✅ **Centralized field definitions** - Easy to maintain
4. ✅ **RCA-friendly features** - Computed automatically for better model learning

**Why This Matters:**
- Prevents "123.4" string vs 123.4 float bugs in calculations
- Ensures distance/ratio computations work correctly
- Makes feature engineering more reliable

In [4]:
import re
import json
import math
import pandas as pd
from statistics import mean, median
from typing import List, Dict, Any, Optional

# ============================================================================
# LABEL MAPPING
# ============================================================================
CAUSE_TO_NUM = {f"C{i}": i for i in range(1, 9)}
NUM_TO_CAUSE = {i: f"C{i}" for i in range(1, 9)}

# ============================================================================
# TYPE CONVERSION UTILITIES
# ============================================================================

def to_float(x) -> Optional[float]:
    """Safely convert to float, handling None, '-', and invalid values"""
    if x is None or x == "" or x == "-" or x == "—":
        return None
    try:
        return float(x)
    except (ValueError, TypeError):
        return None

def to_int(x) -> Optional[int]:
    """Safely convert to int, handling None, '-', and invalid values"""
    if x is None or x == "" or x == "-" or x == "—":
        return None
    try:
        return int(float(x))  # float first to handle "123.0" strings
    except (ValueError, TypeError):
        return None

def sanitize_question_text(q: str) -> str:
    """
    Convert literal '\\n' to real newlines.
    CRITICAL: Don't use unicode decoding or you corrupt '\\boxed'
    """
    if q is None:
        return ""
    return q.replace("\\n", "\n").replace("\r\n", "\n").replace("\r", "\n")

# ============================================================================
# DOMAIN-SPECIFIC CALCULATIONS
# ============================================================================

def haversine_m(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Calculate distance in meters between two lat/lon points"""
    R = 6371000.0  # Earth radius in meters
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlmb = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlmb/2)**2
    return 2 * R * math.asin(math.sqrt(a))

def beamwidth_deg(beam_scenario: str) -> int:
    """
    Map beam scenario to vertical beamwidth:
    - DEFAULT/SCENARIO_1-5: 6 degrees
    - SCENARIO_6-11: 12 degrees
    - SCENARIO_12+: 25 degrees
    """
    s = (beam_scenario or "").upper().strip()
    if s == "DEFAULT":
        return 6

    m = re.match(r"SCENARIO_(\d+)", s)
    if not m:
        return 6

    k = int(m.group(1))
    if 1 <= k <= 5:
        return 6
    elif 6 <= k <= 11:
        return 12
    else:  # 12+
        return 25

def electronic_tilt_deg(digital_tilt) -> float:
    """
    Convert digital tilt to degrees:
    - 255 => 6 degrees (special case)
    - Otherwise: value is degrees
    """
    try:
        v = int(float(digital_tilt))
        return 6.0 if v == 255 else float(v)
    except (ValueError, TypeError):
        return 6.0  # Default fallback

print("✓ Type conversion and domain utilities loaded")

✓ Type conversion and domain utilities loaded


In [5]:
# ============================================================================
# TABLE PARSING
# ============================================================================

def parse_pipe_table(table_text: str) -> List[Dict[str, Any]]:
    """
    Parse pipe-delimited table into list of dicts.
    Handles '-', '—', and empty values as None.
    """
    if not table_text:
        return []

    lines = [ln.strip() for ln in table_text.splitlines() if ln.strip()]
    lines = [ln for ln in lines if "|" in ln]

    if not lines:
        return []

    # Parse header
    header = [h.strip() for h in lines[0].split("|")]

    # Parse rows
    rows = []
    for ln in lines[1:]:
        parts = [p.strip() for p in ln.split("|")]

        # Normalize row length to match header
        if len(parts) < len(header):
            parts += [""] * (len(header) - len(parts))
        elif len(parts) > len(header):
            parts = parts[:len(header)]

        # Build record
        rec = {}
        for k, v in zip(header, parts):
            rec[k] = None if v in ("", "-", "—", "–") else v

        rows.append(rec)

    return rows

print("✓ Table parsing loaded")

✓ Table parsing loaded


In [6]:
# ============================================================================
# COLUMN MAPPING & TYPE CASTING (IMPROVED)
# ============================================================================

# Drive-test data column mappings
DRIVE_MAP = {
    "Timestamp": "timestamp",
    "Longitude": "longitude",
    "Latitude": "latitude",
    "GPS Speed (km/h)": "gps_speed_kmh",
    "5G KPI PCell RF Serving PCI": "serving_pci",
    "5G KPI PCell RF Serving SS-RSRP [dBm]": "ss_rsrp_dbm",
    "5G KPI PCell RF Serving SS-SINR [dB]": "ss_sinr_db",
    "5G KPI PCell Layer2 MAC DL Throughput [Mbps]": "throughput_mbps",
    "5G KPI PCell Layer1 DL RB Num (Including 0)": "dl_rb_num",

    # Neighbor PCIs
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 1 PCI": "nei1_pci",
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 2 PCI": "nei2_pci",
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 3 PCI": "nei3_pci",
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 4 PCI": "nei4_pci",
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 5 PCI": "nei5_pci",

    # Neighbor RSRPs (CRITICAL - these were missing!)
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 1 Filtered Tx BRSRP [dBm]": "nei1_rsrp_dbm",
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 2 Filtered Tx BRSRP [dBm]": "nei2_rsrp_dbm",
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 3 Filtered Tx BRSRP [dBm]": "nei3_rsrp_dbm",
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 4 Filtered Tx BRSRP [dBm]": "nei4_rsrp_dbm",
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 5 Filtered Tx BRSRP [dBm]": "nei5_rsrp_dbm",
}

# Engineering parameters column mappings
ENG_MAP = {
    "gNodeB ID": "gnodeb_id",
    "Cell ID": "cell_id",
    "Longitude": "longitude",
    "Latitude": "latitude",
    "Mechanical Azimuth": "mechanical_azimuth",
    "Mechanical Downtilt": "mechanical_downtilt",
    "Digital Tilt": "digital_tilt",
    "Digital Azimuth": "digital_azimuth",
    "Beam Scenario": "beam_scenario",
    "Height": "height",
    "PCI": "pci",
    "TxRx Mode": "txrx_mode",
    "Max Transmit Power": "max_tx_power",
    "Antenna Model": "antenna_model",
}

# Define fields by type for proper casting
FLOAT_FIELDS = [
    "longitude", "latitude", "gps_speed_kmh",
    "ss_rsrp_dbm", "ss_sinr_db", "throughput_mbps", "dl_rb_num",
    "mechanical_downtilt", "digital_tilt", "height", "max_tx_power",
    "mechanical_azimuth", "digital_azimuth",
    # CRITICAL: Add neighbor RSRP fields
    "nei1_rsrp_dbm", "nei2_rsrp_dbm", "nei3_rsrp_dbm",
    "nei4_rsrp_dbm", "nei5_rsrp_dbm",
]

INT_FIELDS = [
    "pci", "serving_pci",
    "nei1_pci", "nei2_pci", "nei3_pci", "nei4_pci", "nei5_pci",
]

def normalize_rows(rows: List[Dict], mapping: Dict[str, str]) -> List[Dict]:
    """
    Normalize column names and apply type casting.
    This ensures all numeric operations work correctly.
    """
    normalized = []

    for r in rows:
        nr = {}

        # Map column names
        for k, v in r.items():
            normalized_key = mapping.get(k)
            if normalized_key is None:
                continue  # Skip unmapped columns
            nr[normalized_key] = v

        # Type casting - CRITICAL for numeric operations
        for field in FLOAT_FIELDS:
            if field in nr:
                nr[field] = to_float(nr[field])

        for field in INT_FIELDS:
            if field in nr:
                nr[field] = to_int(nr[field])

        normalized.append(nr)

    return normalized

print("✓ Column mappings and type casting configured")
print(f"  - Float fields: {len(FLOAT_FIELDS)}")
print(f"  - Int fields: {len(INT_FIELDS)}")
print(f"  - Drive-test columns mapped: {len(DRIVE_MAP)}")
print(f"  - Engineering columns mapped: {len(ENG_MAP)}")

✓ Column mappings and type casting configured
  - Float fields: 18
  - Int fields: 7
  - Drive-test columns mapped: 19
  - Engineering columns mapped: 14


In [7]:
# ============================================================================
# RCA-FRIENDLY FEATURE ENGINEERING
# ============================================================================

def compute_rca_features(drive_rows: List[Dict], eng_rows: List[Dict]) -> Dict[str, Any]:
    """
    Compute derived features that help the model learn root cause patterns.
    This is the KEY to good RCA performance!

    Returns dict with all computed features.
    """
    features = {}

    # -------------------------------------------------------------------------
    # A) Throughput-drop signature (C8 indicator)
    # -------------------------------------------------------------------------
    tps = [r["throughput_mbps"] for r in drive_rows if r.get("throughput_mbps") is not None]

    if tps:
        features["tp_min_mbps"] = min(tps)
        features["tp_mean_mbps"] = mean(tps)
        features["tp_max_mbps"] = max(tps)
        features["tp_drop_ratio"] = sum(1 for x in tps if x < 600) / len(tps)
        features["tp_samples_below_600"] = sum(1 for x in tps if x < 600)
    else:
        features["tp_min_mbps"] = None
        features["tp_mean_mbps"] = None
        features["tp_max_mbps"] = None
        features["tp_drop_ratio"] = None
        features["tp_samples_below_600"] = 0

    # -------------------------------------------------------------------------
    # B) Resource constraint (C8)
    # -------------------------------------------------------------------------
    rbs = [r["dl_rb_num"] for r in drive_rows if r.get("dl_rb_num") is not None]

    if rbs:
        features["rb_mean"] = mean(rbs)
        features["rb_min"] = min(rbs)
        features["rb_max"] = max(rbs)
        features["rb_below_160_flag"] = features["rb_mean"] < 160 if features["rb_mean"] else False
    else:
        features["rb_mean"] = None
        features["rb_min"] = None
        features["rb_max"] = None
        features["rb_below_160_flag"] = False

    # -------------------------------------------------------------------------
    # C) Speed rule (C7)
    # -------------------------------------------------------------------------
    speeds = [r["gps_speed_kmh"] for r in drive_rows if r.get("gps_speed_kmh") is not None]

    if speeds:
        features["speed_max_kmh"] = max(speeds)
        features["speed_mean_kmh"] = mean(speeds)
        features["speed_above_40_flag"] = features["speed_max_kmh"] > 40 if features["speed_max_kmh"] else False
    else:
        features["speed_max_kmh"] = None
        features["speed_mean_kmh"] = None
        features["speed_above_40_flag"] = False

    # -------------------------------------------------------------------------
    # D) Handover / mobility (C5)
    # -------------------------------------------------------------------------
    serving_pcis = [r["serving_pci"] for r in drive_rows if r.get("serving_pci") is not None]

    handover_count = 0
    for i in range(1, len(serving_pcis)):
        if serving_pcis[i] != serving_pcis[i - 1]:
            handover_count += 1

    features["handover_count"] = handover_count
    features["frequent_handover_flag"] = handover_count >= 3  # Threshold for "frequent"

    # -------------------------------------------------------------------------
    # E) PCI mod 30 collision (C6)
    # -------------------------------------------------------------------------
    neighbor_pcis = []
    for r in drive_rows:
        for k in ["nei1_pci", "nei2_pci", "nei3_pci", "nei4_pci", "nei5_pci"]:
            v = r.get(k)
            if v is not None:
                neighbor_pcis.append(v)

    mod30_collision = False
    if serving_pcis and neighbor_pcis:
        serving_pci = serving_pcis[-1]  # Use last serving PCI
        mod30_collision = any((n % 30) == (serving_pci % 30) for n in neighbor_pcis)

    features["pci_mod30_collision"] = mod30_collision

    # -------------------------------------------------------------------------
    # F) Coverage distance / overshoot (C2)
    # -------------------------------------------------------------------------
    pci_to_cell = {c["pci"]: c for c in eng_rows if c.get("pci") is not None}

    distances = []
    for r in drive_rows:
        spci = r.get("serving_pci")
        cell = pci_to_cell.get(spci)

        if not cell:
            continue

        if (r.get("latitude") is None or r.get("longitude") is None or
            cell.get("latitude") is None or cell.get("longitude") is None):
            continue

        dist = haversine_m(
            r["latitude"], r["longitude"],
            cell["latitude"], cell["longitude"]
        )
        distances.append(dist)

    if distances:
        features["dist_median_m"] = median(distances)
        features["dist_p95_m"] = sorted(distances)[int(0.95 * (len(distances) - 1))]
        features["dist_max_m"] = max(distances)
        features["overshoot_flag"] = features["dist_p95_m"] > 1000 if features["dist_p95_m"] else False
    else:
        features["dist_median_m"] = None
        features["dist_p95_m"] = None
        features["dist_max_m"] = None
        features["overshoot_flag"] = False

    # -------------------------------------------------------------------------
    # G) Downtilt / weak far coverage (C1)
    # -------------------------------------------------------------------------
    if serving_pcis:
        cell = pci_to_cell.get(serving_pcis[-1])

        if cell:
            mech_tilt = cell.get("mechanical_downtilt") or 0.0
            elec_tilt = electronic_tilt_deg(cell.get("digital_tilt"))
            total_tilt = float(mech_tilt) + float(elec_tilt)

            vbw = beamwidth_deg(cell.get("beam_scenario", "DEFAULT"))
            tilt_to_beamwidth_ratio = total_tilt / vbw if vbw > 0 else 0.0

            features["serving_mechanical_tilt_deg"] = mech_tilt
            features["serving_electronic_tilt_deg"] = elec_tilt
            features["serving_total_tilt_deg"] = total_tilt
            features["serving_vertical_beamwidth_deg"] = vbw
            features["tilt_to_beamwidth_ratio"] = tilt_to_beamwidth_ratio
            features["large_tilt_flag"] = total_tilt > 15  # Threshold for "too large"
        else:
            features["serving_mechanical_tilt_deg"] = None
            features["serving_electronic_tilt_deg"] = None
            features["serving_total_tilt_deg"] = None
            features["serving_vertical_beamwidth_deg"] = None
            features["tilt_to_beamwidth_ratio"] = None
            features["large_tilt_flag"] = False
    else:
        features["serving_mechanical_tilt_deg"] = None
        features["serving_electronic_tilt_deg"] = None
        features["serving_total_tilt_deg"] = None
        features["serving_vertical_beamwidth_deg"] = None
        features["tilt_to_beamwidth_ratio"] = None
        features["large_tilt_flag"] = False

    # -------------------------------------------------------------------------
    # H) Overlapping coverage (C4)
    # -------------------------------------------------------------------------
    # Count strong neighbors (RSRP within 5dB of strongest and > -95dBm)
    neighbor_rsrps = []
    for r in drive_rows:
        for k in ["nei1_rsrp_dbm", "nei2_rsrp_dbm", "nei3_rsrp_dbm",
                  "nei4_rsrp_dbm", "nei5_rsrp_dbm"]:
            v = r.get(k)
            if v is not None:
                neighbor_rsrps.append(v)

    if neighbor_rsrps:
        strongest_nei = max(neighbor_rsrps)
        strong_neighbors = [
            rsrp for rsrp in neighbor_rsrps
            if rsrp > -95 and abs(rsrp - strongest_nei) <= 5
        ]
        features["strong_neighbor_count"] = len(strong_neighbors)
        features["overlap_flag"] = len(strong_neighbors) >= 3  # Multiple strong neighbors
    else:
        features["strong_neighbor_count"] = 0
        features["overlap_flag"] = False

    return features

print("✓ RCA feature engineering ready")
print("  Features computed:")
print("    - Throughput metrics (tp_min, tp_mean, tp_drop_ratio)")
print("    - Resource blocks (rb_mean, rb_below_160_flag)")
print("    - Speed (speed_max, speed_above_40_flag)")
print("    - Handovers (handover_count, frequent_handover_flag)")
print("    - PCI mod 30 collisions")
print("    - Distance/overshoot (dist_median, dist_p95, overshoot_flag)")
print("    - Tilt analysis (total_tilt, tilt_to_beamwidth_ratio)")
print("    - Overlap detection (strong_neighbor_count, overlap_flag)")

✓ RCA feature engineering ready
  Features computed:
    - Throughput metrics (tp_min, tp_mean, tp_drop_ratio)
    - Resource blocks (rb_mean, rb_below_160_flag)
    - Speed (speed_max, speed_above_40_flag)
    - Handovers (handover_count, frequent_handover_flag)
    - PCI mod 30 collisions
    - Distance/overshoot (dist_median, dist_p95, overshoot_flag)
    - Tilt analysis (total_tilt, tilt_to_beamwidth_ratio)
    - Overlap detection (strong_neighbor_count, overlap_flag)


In [8]:
# ============================================================================
# ENHANCED QUESTION FORMATTING (Q&A Format)
# ============================================================================

def format_value(v) -> str:
    """Format value for display in prompt"""
    if v is None:
        return "N/A"
    if isinstance(v, bool):
        return "Yes" if v else "No"
    if isinstance(v, int):
        return str(v)
    if isinstance(v, float):
        return f"{v:.2f}"
    return str(v)

def format_features_text(features: Dict) -> str:
    """
    Format engineered features as a readable text block.
    This will be stored in a separate column.
    """
    feature_text = "\n".join([
        "**Derived Features Summary:**",
        "",
        "**Throughput Metrics:**",
        f"  - Min: {format_value(features.get('tp_min_mbps'))} Mbps",
        f"  - Mean: {format_value(features.get('tp_mean_mbps'))} Mbps",
        f"  - Drop ratio (<600 Mbps): {format_value(features.get('tp_drop_ratio'))}",
        f"  - Samples below 600: {format_value(features.get('tp_samples_below_600'))}",
        "",
        "**Resource Blocks:**",
        f"  - Mean RB: {format_value(features.get('rb_mean'))}",
        f"  - Min RB: {format_value(features.get('rb_min'))}",
        f"  - Max RB: {format_value(features.get('rb_max'))}",
        f"  - Below 160 threshold: {format_value(features.get('rb_below_160_flag'))}",
        "",
        "**Speed:**",
        f"  - Max speed: {format_value(features.get('speed_max_kmh'))} km/h",
        f"  - Mean speed: {format_value(features.get('speed_mean_kmh'))} km/h",
        f"  - Exceeds 40 km/h: {format_value(features.get('speed_above_40_flag'))}",
        "",
        "**Handovers:**",
        f"  - Count: {format_value(features.get('handover_count'))}",
        f"  - Frequent handovers (≥3): {format_value(features.get('frequent_handover_flag'))}",
        "",
        "**PCI Collision:**",
        f"  - Mod 30 collision detected: {format_value(features.get('pci_mod30_collision'))}",
        "",
        "**Coverage Distance:**",
        f"  - Median distance: {format_value(features.get('dist_median_m'))} m",
        f"  - 95th percentile: {format_value(features.get('dist_p95_m'))} m",
        f"  - Max distance: {format_value(features.get('dist_max_m'))} m",
        f"  - Overshoot (>1km): {format_value(features.get('overshoot_flag'))}",
        "",
        "**Tilt Analysis:**",
        f"  - Mechanical tilt: {format_value(features.get('serving_mechanical_tilt_deg'))}°",
        f"  - Electronic tilt: {format_value(features.get('serving_electronic_tilt_deg'))}°",
        f"  - Total tilt: {format_value(features.get('serving_total_tilt_deg'))}°",
        f"  - Vertical beamwidth: {format_value(features.get('serving_vertical_beamwidth_deg'))}°",
        f"  - Tilt/beamwidth ratio: {format_value(features.get('tilt_to_beamwidth_ratio'))}",
        f"  - Large tilt (>15°): {format_value(features.get('large_tilt_flag'))}",
        "",
        "**Overlap Detection:**",
        f"  - Strong neighbors count: {format_value(features.get('strong_neighbor_count'))}",
        f"  - Overlap suspected (≥3): {format_value(features.get('overlap_flag'))}",
    ])
    
    return feature_text

def format_enhanced_question(original_question: str, features_text: str) -> str:
    """
    Combine the original question with engineered features.
    This creates the enhanced question column WITHOUT the answer.
    """
    enhanced_question = f"""{original_question}

{features_text}"""
    
    return enhanced_question

print("✓ Q&A format functions ready")
print("  - format_features_text(): Formats features into readable text")
print("  - format_enhanced_question(): Combines original question + features")


✓ Q&A format functions ready
  - format_features_text(): Formats features into readable text
  - format_enhanced_question(): Combines original question + features


In [9]:
# ============================================================================
# MAIN PREPROCESSING FUNCTION (DataFrame-Based)
# ============================================================================

def preprocess_row(row: Dict) -> Optional[Dict]:
    """
    Main preprocessing pipeline that returns structured data for DataFrame.
    
    Returns a dict with:
    - ID: Sample identifier
    - original_question: The original question text (sanitized)
    - features_text: Formatted engineered features
    - enhanced_question: original_question + features_text
    - answer: The answer label (C1-C8)
    - features_dict: Raw feature values as dict (for analysis)
    
    Returns None if row cannot be processed.
    """
    # Sanitize question text
    question_text = sanitize_question_text(row["question"])
    label = str(row["answer"]).strip().upper()

    # Validate label
    if label not in CAUSE_TO_NUM:
        return None

    # Extract table sections
    drive_match = re.search(r"User plane drive test data as follows[:：]\s*", question_text)
    eng_match = re.search(r"Engeneering parameters data as follows[:：]\s*", question_text)

    if not drive_match or not eng_match or eng_match.start() <= drive_match.end():
        return None

    drive_text = question_text[drive_match.end():eng_match.start()].strip()
    eng_text = question_text[eng_match.end():].strip()

    # Parse tables
    drive_raw = parse_pipe_table(drive_text)
    eng_raw = parse_pipe_table(eng_text)

    if not drive_raw or not eng_raw:
        return None

    # Normalize with type casting (CRITICAL!)
    drive_rows = normalize_rows(drive_raw, DRIVE_MAP)
    eng_rows = normalize_rows(eng_raw, ENG_MAP)

    # Compute RCA features
    features = compute_rca_features(drive_rows, eng_rows)

    # Format features as text
    features_text = format_features_text(features)

    # Enhanced question = original + features (NO answer)
    enhanced_question = format_enhanced_question(question_text, features_text)

    # Return structured data for DataFrame
    return {
        "ID": row["ID"],
        "original_question": question_text,
        "features_text": features_text,
        "enhanced_question": enhanced_question,
        "answer": label,
        "features_dict": features  # Keep raw dict for analysis
    }

print("✓ DataFrame-based preprocessing pipeline ready")
print("  Returns dict with columns:")
print("    - ID: Sample identifier")
print("    - original_question: Original question text")
print("    - features_text: Formatted engineered features")
print("    - enhanced_question: Question + features combined")
print("    - answer: Root cause label (C1-C8)")
print("    - features_dict: Raw feature values (for analysis)")


✓ DataFrame-based preprocessing pipeline ready
  Returns dict with columns:
    - ID: Sample identifier
    - original_question: Original question text
    - features_text: Formatted engineered features
    - enhanced_question: Question + features combined
    - answer: Root cause label (C1-C8)
    - features_dict: Raw feature values (for analysis)


## 🚀 Apply Enhanced Preprocessing

Now let's process the training data with the improved pipeline that includes:
- Proper type casting (no more string/number bugs)
- RCA-friendly feature engineering
- Enhanced chat formatting

In [10]:
# Process all training samples into DataFrame format
print("=" * 70)
print("PROCESSING TRAINING DATA INTO DATAFRAME FORMAT")
print("=" * 70)
print(f"Total samples to process: {len(df_train):,}\n")

processed_records = []
failed_count = 0

for idx, row_dict in enumerate(df_train.to_dict(orient="records")):
    if idx % 10000 == 0 and idx > 0:
        print(f"  Processed {idx:,} samples... ({len(processed_records):,} successful)")

    result = preprocess_row(row_dict)

    if result is not None:
        processed_records.append(result)
    else:
        failed_count += 1

print(f"\n{'='*70}")
print(f"✓ Processing complete!")
print(f"  - Successfully processed: {len(processed_records):,}")
print(f"  - Failed: {failed_count:,}")
print(f"  - Success rate: {100 * len(processed_records) / len(df_train):.1f}%")
print(f"{'='*70}\n")

# Create DataFrame with structured columns
df_processed = pd.DataFrame(processed_records)

print("✓ Created DataFrame with columns:")
for col in df_processed.columns:
    if col != 'features_dict':  # Don't show dict column details
        print(f"  - {col}")
print(f"\nDataFrame shape: {df_processed.shape}")
print(f"Memory usage: {df_processed.memory_usage(deep=True).sum() / 1024**2:.1f} MB")


PROCESSING TRAINING DATA INTO DATAFRAME FORMAT
Total samples to process: 2,400


✓ Processing complete!
  - Successfully processed: 2,400
  - Failed: 0
  - Success rate: 100.0%

✓ Created DataFrame with columns:
  - ID
  - original_question
  - features_text
  - enhanced_question
  - answer

DataFrame shape: (2400, 6)
Memory usage: 51.3 MB

✓ Processing complete!
  - Successfully processed: 2,400
  - Failed: 0
  - Success rate: 100.0%

✓ Created DataFrame with columns:
  - ID
  - original_question
  - features_text
  - enhanced_question
  - answer

DataFrame shape: (2400, 6)
Memory usage: 51.3 MB


In [11]:
# Display DataFrame info
print("=" * 70)
print("DATAFRAME STRUCTURE")
print("=" * 70)
display(df_processed.head(3))

print("\n" + "=" * 70)
print("COLUMN DETAILS")
print("=" * 70)
print(f"Total rows: {len(df_processed):,}")
print(f"\nColumn types:")
for col in df_processed.columns:
    if col != 'features_dict':
        print(f"  {col}: {df_processed[col].dtype}")

print("\n" + "=" * 70)
print("ANSWER DISTRIBUTION")
print("=" * 70)
print(df_processed['answer'].value_counts().sort_index())

DATAFRAME STRUCTURE


,ID,original_question,features_text,enhanced_question,answer,features_dict
0,ID_1P7PJMPV0R,Analyze the 5G wireless network drive-test use...,**Derived Features Summary:**\n\n**Throughput ...,Analyze the 5G wireless network drive-test use...,C2,"{'tp_min_mbps': 334.0, 'tp_mean_mbps': 847.792..."
1,ID_8B1D1TUTFA,Analyze the 5G wireless network drive-test use...,**Derived Features Summary:**\n\n**Throughput ...,Analyze the 5G wireless network drive-test use...,C1,"{'tp_min_mbps': 388.58, 'tp_mean_mbps': 850.05..."
2,ID_IGGXMA9GZH,Analyze the 5G wireless network drive-test use...,**Derived Features Summary:**\n\n**Throughput ...,Analyze the 5G wireless network drive-test use...,C2,"{'tp_min_mbps': 258.08, 'tp_mean_mbps': 671.73..."



COLUMN DETAILS
Total rows: 2,400

Column types:
  ID: object
  original_question: object
  features_text: object
  enhanced_question: object
  answer: object

ANSWER DISTRIBUTION
answer
C1    264
C2    320
C3    330
C4    283
C5    352
C6    225
C7    349
C8    277
Name: count, dtype: int64


In [19]:
# Show example of each column
print("=" * 70)
print("EXAMPLE: ORIGINAL QUESTION (first 800 chars)")
print("=" * 70)
print(df_processed['original_question'].iloc[0][:800] + "...")

print("\n" + "=" * 70)
print("EXAMPLE: FEATURES TEXT")
print("=" * 70)
print(df_processed['features_text'].iloc[0])

print("\n" + "=" * 70)
print("EXAMPLE: ENHANCED QUESTION (first 1000 chars)")
print("=" * 70)
print(df_processed['enhanced_question'].iloc[0] + "...")

print("\n" + "=" * 70)
print("EXAMPLE: ANSWER")
print("=" * 70)
print(f"Answer: {df_processed['answer'].iloc[0]}")

EXAMPLE: ORIGINAL QUESTION (first 800 chars)
Analyze the 5G wireless network drive-test user plane data and engineering parameters.
Identify the reason for the throughput dropping below 600Mbps in certain road sections.
From the following 8 potential root causes, select the most likely one and enclose its number in \boxed{{}} in the final answer.

C1: The serving cell's downtilt angle is too large, causing weak coverage at the far end.
C2: The serving cell's coverage distance exceeds 1km, resulting in over-shooting.
C3: A neighboring cell provides higher throughput.
C4: Non-colocated co-frequency neighboring cells cause severe overlapping coverage.
C5: Frequent handovers degrade performance.
C6: Neighbor cell and serving cell have the same PCI mod 30, leading to interference.
C7: Test vehicle speed exceeds 40km/h, impacting user thro...

EXAMPLE: FEATURES TEXT
**Derived Features Summary:**

**Throughput Metrics:**
  - Min: 334.00 Mbps
  - Mean: 847.79 Mbps
  - Drop ratio (<600 Mbps): 0

In [13]:
# Analyze the computed features
print("=" * 70)
print("FEATURE STATISTICS ANALYSIS")
print("=" * 70)

# Extract all features for analysis
all_features = df_processed['features_dict'].tolist()

# Analyze key RCA indicators
feature_stats = {}
for key in all_features[0].keys():
    values = [f.get(key) for f in all_features if f.get(key) is not None]

    if len(values) > 0:
        if isinstance(values[0], bool):
            feature_stats[key] = {
                "type": "boolean",
                "true_count": sum(values),
                "false_count": len(values) - sum(values),
                "true_pct": 100 * sum(values) / len(values)
            }
        elif isinstance(values[0], (int, float)):
            feature_stats[key] = {
                "type": "numeric",
                "min": min(values),
                "max": max(values),
                "mean": sum(values) / len(values),
                "count": len(values)
            }

# Display key statistics
print("\n🎯 RCA INDICATOR STATISTICS:\n")

print("**Throughput Issues (C8 related):**")
if "tp_drop_ratio" in feature_stats:
    s = feature_stats["tp_drop_ratio"]
    print(f"  Drop ratio: mean={s['mean']:.2f}, min={s['min']:.2f}, max={s['max']:.2f}")
if "rb_below_160_flag" in feature_stats:
    s = feature_stats["rb_below_160_flag"]
    print(f"  RB below 160: {s['true_pct']:.1f}% of samples")

print("\n**Speed Issues (C7):**")
if "speed_above_40_flag" in feature_stats:
    s = feature_stats["speed_above_40_flag"]
    print(f"  Speed > 40 km/h: {s['true_pct']:.1f}% of samples")

print("\n**Handover Issues (C5):**")
if "frequent_handover_flag" in feature_stats:
    s = feature_stats["frequent_handover_flag"]
    print(f"  Frequent handovers: {s['true_pct']:.1f}% of samples")

print("\n**PCI Collision (C6):**")
if "pci_mod30_collision" in feature_stats:
    s = feature_stats["pci_mod30_collision"]
    print(f"  PCI mod 30 collision: {s['true_pct']:.1f}% of samples")

print("\n**Overshoot (C2):**")
if "overshoot_flag" in feature_stats:
    s = feature_stats["overshoot_flag"]
    print(f"  Overshoot (>1km): {s['true_pct']:.1f}% of samples")

print("\n**Tilt Issues (C1):**")
if "large_tilt_flag" in feature_stats:
    s = feature_stats["large_tilt_flag"]
    print(f"  Large tilt (>15°): {s['true_pct']:.1f}% of samples")

print("\n**Overlap (C4):**")
if "overlap_flag" in feature_stats:
    s = feature_stats["overlap_flag"]
    print(f"  Overlap detected: {s['true_pct']:.1f}% of samples")

print(f"\n{'='*70}")

FEATURE STATISTICS ANALYSIS

🎯 RCA INDICATOR STATISTICS:

**Throughput Issues (C8 related):**
  Drop ratio: mean=0.39, min=0.00, max=0.40
  RB below 160: 1.7% of samples

**Speed Issues (C7):**
  Speed > 40 km/h: 14.5% of samples

**Handover Issues (C5):**
  Frequent handovers: 14.7% of samples

**PCI Collision (C6):**
  PCI mod 30 collision: 100.0% of samples

**Overshoot (C2):**
  Overshoot (>1km): 13.3% of samples

**Tilt Issues (C1):**
  Large tilt (>15°): 36.0% of samples

**Overlap (C4):**
  Overlap detected: 68.5% of samples



## 💾 Save Enhanced Training Data

Save the DataFrame in multiple formats:
1. **CSV** - Easy to inspect and use with pandas
2. **Parquet** - Efficient storage for large datasets
3. **HuggingFace Dataset** - For Transformers training

In [15]:
# Save as CSV (without features_dict column for readability)
csv_path = "qwen_rca_train_enhanced.csv"
df_csv = df_processed.drop(columns=['features_dict'])

print(f"Saving to {csv_path}...")
df_csv.to_csv(csv_path, index=False, encoding='utf-8')

print(f"✓ Saved {len(df_csv):,} samples to {csv_path}")
print(f"  File size: {Path(csv_path).stat().st_size / 1024 / 1024:.1f} MB")
print(f"  Columns: {', '.join(df_csv.columns.tolist())}")

# Save as Parquet (more efficient, includes all columns)
parquet_path = "qwen_rca_train_enhanced.parquet"
print(f"\nSaving to {parquet_path}...")
df_processed.to_parquet(parquet_path, index=False, engine='pyarrow')

print(f"✓ Saved {len(df_processed):,} samples to {parquet_path}")
print(f"  File size: {Path(parquet_path).stat().st_size / 1024 / 1024:.1f} MB")
print(f"  Columns: {', '.join(df_processed.columns.tolist())}")

print("\n✓ Data saved successfully!")
print("\nColumn summary:")
print("  - ID: Sample identifier")
print("  - original_question: Original question from training data")
print("  - features_text: Formatted engineered features")
print("  - enhanced_question: Question + features (for model input)")
print("  - answer: Root cause label (C1-C8)")
print("  - features_dict: Raw feature values (parquet only)")


Saving to qwen_rca_train_enhanced.csv...
✓ Saved 2,400 samples to qwen_rca_train_enhanced.csv
  File size: 24.4 MB
  Columns: ID, original_question, features_text, enhanced_question, answer

Saving to qwen_rca_train_enhanced.parquet...
✓ Saved 2,400 samples to qwen_rca_train_enhanced.parquet
  File size: 6.0 MB
  Columns: ID, original_question, features_text, enhanced_question, answer, features_dict

✓ Data saved successfully!

Column summary:
  - ID: Sample identifier
  - original_question: Original question from training data
  - features_text: Formatted engineered features
  - enhanced_question: Question + features (for model input)
  - answer: Root cause label (C1-C8)
  - features_dict: Raw feature values (parquet only)
✓ Saved 2,400 samples to qwen_rca_train_enhanced.csv
  File size: 24.4 MB
  Columns: ID, original_question, features_text, enhanced_question, answer

Saving to qwen_rca_train_enhanced.parquet...
✓ Saved 2,400 samples to qwen_rca_train_enhanced.parquet
  File size: 6

In [16]:
# Create HuggingFace Dataset for training (Q&A format)
from datasets import Dataset

# Prepare training data with Q&A format
# We'll use 'enhanced_question' as input and 'answer' as label
train_data = []
for _, row in df_processed.iterrows():
    train_data.append({
        "question": row["enhanced_question"],  # Question with features
        "answer": row["answer"],  # Just the label (C1-C8)
        "id": row["ID"]
    })

hf_dataset = Dataset.from_list(train_data)

print(f"\n✓ Created HuggingFace dataset: {len(hf_dataset):,} samples")
print(f"  Columns: {hf_dataset.column_names}")
print(f"  Features: {hf_dataset.features}")

# Save to disk for quick loading later
hf_dataset.save_to_disk("qwen_rca_dataset_enhanced")
print(f"✓ Saved HuggingFace dataset to 'qwen_rca_dataset_enhanced/'")

# Create a train/validation split (90/10)
train_test = hf_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = train_test["train"]
val_dataset = train_test["test"]

print(f"\n📊 Dataset split:")
print(f"  Training: {len(train_dataset):,} samples")
print(f"  Validation: {len(val_dataset):,} samples")

print("\n✓ Dataset format:")
print("  - question: Enhanced question (original + features)")
print("  - answer: Root cause label (C1-C8)")
print("  - id: Sample identifier")



✓ Created HuggingFace dataset: 2,400 samples
  Columns: ['question', 'answer', 'id']
  Features: {'question': Value('string'), 'answer': Value('string'), 'id': Value('string')}


Saving the dataset (1/1 shards): 100%|██████████| 2400/2400 [00:00<00:00, 286602.22 examples/s]

✓ Saved HuggingFace dataset to 'qwen_rca_dataset_enhanced/'

📊 Dataset split:
  Training: 2,160 samples
  Validation: 240 samples

✓ Dataset format:
  - question: Enhanced question (original + features)
  - answer: Root cause label (C1-C8)
  - id: Sample identifier


In [17]:
# Preview the dataset structure
print("Dataset structure:")
display(train_dataset)
print("\nFirst sample:")
print(f"ID: {train_dataset['id'][0]}")
print(f"\nQuestion (first 500 chars):\n{train_dataset['question'][0][:500]}...")
print(f"\nAnswer: {train_dataset['answer'][0]}")


Dataset structure:


Dataset({
    features: ['question', 'answer', 'id'],
    num_rows: 2160
})


First sample:
ID: ID_AOAWXFSCEK

Question (first 500 chars):
Analyze the 5G wireless network drive-test user plane data and engineering parameters.
Identify the reason for the throughput dropping below 600Mbps in certain road sections.
From the following 8 potential root causes, select the most likely one and enclose its number in \boxed{{}} in the final answer.

C1: The serving cell's downtilt angle is too large, causing weak coverage at the far end.
C2: The serving cell's coverage distance exceeds 1km, resulting in over-shooting.
C3: A neighboring cell ...

Answer: C4


In [20]:
# Check a specific sample
sample_idx = 2
print(f"Sample {sample_idx}:")
print(f"ID: {train_dataset['id'][sample_idx]}")
print(f"\nQuestion (first 800 chars):\n{train_dataset['question'][sample_idx]}...")
print(f"\nAnswer: {train_dataset['answer'][sample_idx]}")


Sample 2:
ID: ID_ZRWTXDOY5M

Question (first 800 chars):
Analyze the 5G wireless network drive-test user plane data and engineering parameters.
Identify the reason for the throughput dropping below 600Mbps in certain road sections.
From the following 8 potential root causes, select the most likely one and enclose its number in \boxed{{}} in the final answer.

C1: The serving cell's downtilt angle is too large, causing weak coverage at the far end.
C2: The serving cell's coverage distance exceeds 1km, resulting in over-shooting.
C3: A neighboring cell provides higher throughput.
C4: Non-colocated co-frequency neighboring cells cause severe overlapping coverage.
C5: Frequent handovers degrade performance.
C6: Neighbor cell and serving cell have the same PCI mod 30, leading to interference.
C7: Test vehicle speed exceeds 40km/h, impacting user throughput.
C8: Average scheduled RBs are below 160, affecting throughput.

Given:
- The default electronic downtilt value is 255, representing a d

In [21]:
# Verify dataset type
print(f"Dataset type: {type(train_dataset)}")
print(f"Number of samples: {len(train_dataset)}")
print(f"Features: {train_dataset.features}")


Dataset type: <class 'datasets.arrow_dataset.Dataset'>
Number of samples: 2160
Features: {'question': Value('string'), 'answer': Value('string'), 'id': Value('string')}


In [22]:
# Step 2: Imports
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, TaskType
import pandas as pd
from datasets import Dataset
import torch

In [23]:
# Step 4: Load Tokenizer and Model (4-bit quant on GPU; CPU fallback)

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

print(f"Loading tokenizer: {model_name}")
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

# Ensure we have a pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"✓ Tokenizer pad token: {tokenizer.pad_token}")

use_cuda = torch.cuda.is_available()

if use_cuda:
    print("CUDA detected → loading 4-bit quantized model")

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,   # use torch.float16 if your GPU doesn't support bf16
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    )

    # Prepare for k-bit LoRA/QLoRA training
    model = prepare_model_for_kbit_training(model)

    print("✓ Model loaded with 4-bit quantization")
    print(f"✓ GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

else:
    print("CUDA not detected → loading model on CPU (not recommended for training)")

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float32,
        device_map="cpu",
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    )

    print("✓ Model loaded on CPU")


Loading tokenizer: Qwen/Qwen2.5-1.5B-Instruct
✓ Tokenizer pad token: <|endoftext|>
CUDA not detected → loading model on CPU (not recommended for training)
✓ Tokenizer pad token: <|endoftext|>
CUDA not detected → loading model on CPU (not recommended for training)


`torch_dtype` is deprecated! Use `dtype` instead!


✓ Model loaded on CPU


In [24]:
# Step 5: Apply LoRA for efficient fine-tuning

import torch
from peft import LoraConfig, TaskType, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)

print("✓ LoRA applied successfully")
model.print_trainable_parameters()

if torch.cuda.is_available():
    print(f"✓ GPU memory after LoRA: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")


✓ LoRA applied successfully
trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


In [27]:
# Step 6: Tokenize with Q&A Format (using chat template)
def tokenize_qa_with_chat_template(examples):
    """
    Tokenize Q&A data using Qwen's chat template.
    Converts question/answer pairs into the chat format that Qwen expects.
    """
    texts = []
    
    # Convert each Q&A pair to chat format
    for question, answer in zip(examples["question"], examples["answer"]):
        # Create chat messages
        messages = [
            {
                "role": "system",
                "content": "You are a senior 5G RAN optimization engineer. Analyze the provided network data and identify the most likely root cause for throughput degradation. Provide your answer as the root cause label (C1-C8)."
            },
            {
                "role": "user",
                "content": question
            },
            {
                "role": "assistant",
                "content": f"Based on the analysis, the root cause is: {answer}"
            }
        ]
        
        # Apply chat template
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        texts.append(text)
    
    # Tokenize all texts in batch
    tokenized = tokenizer(
        texts,
        truncation=True,
        padding=False,  # Data collator will handle padding
        max_length=1024*3,
        return_tensors=None
    )
    
    # Copy input_ids to labels for causal LM training
    tokenized["labels"] = tokenized["input_ids"].copy()
    
    return tokenized

print("Tokenizing datasets with Q&A chat template...")
train_dataset_tokenized = train_dataset.map(
    tokenize_qa_with_chat_template,
    batched=True,
    remove_columns=train_dataset.column_names,
    desc="Tokenizing train"
)
val_dataset_tokenized = val_dataset.map(
    tokenize_qa_with_chat_template,
    batched=True,
    remove_columns=val_dataset.column_names,
    desc="Tokenizing validation"
)

print(f"✓ Tokenized {len(train_dataset_tokenized)} training samples")
print(f"✓ Tokenized {len(val_dataset_tokenized)} validation samples")
print(f"Sample tokenized output keys: {train_dataset_tokenized.column_names}")

# Check token length statistics
token_lengths = [len(x) for x in train_dataset_tokenized["input_ids"]]
print(f"\nToken length statistics:")
print(f"  Min: {min(token_lengths)}")
print(f"  Max: {max(token_lengths)}")
print(f"  Mean: {sum(token_lengths) / len(token_lengths):.1f}")
print(f"  Median: {sorted(token_lengths)[len(token_lengths)//2]}")


Tokenizing datasets with Q&A chat template...


Tokenizing validation: 100%|██████████| 240/240 [00:00<00:00, 816.52 examples/s]

✓ Tokenized 2160 training samples
✓ Tokenized 240 validation samples
Sample tokenized output keys: ['input_ids', 'attention_mask', 'labels']

Token length statistics:
  Min: 2427
  Max: 3072
  Mean: 2769.3
  Median: 2775


In [28]:
import torch
from transformers import DataCollatorForLanguageModeling, TrainingArguments

# ---------------------------
# Data collator (causal LM)
# ---------------------------
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

# ---------------------------
# Training hyperparams setup
# ---------------------------
output_dir = "./qwen_telco_finetuned"

if torch.cuda.is_available():
    per_device_batch = 2
    gradient_accum = 4  # effective batch = per_device_batch * gradient_accum
    use_bf16 = torch.cuda.is_bf16_supported()
    use_fp16 = not use_bf16
    optim_choice = "paged_adamw_8bit"
    device_name = torch.cuda.get_device_name(0)
else:
    per_device_batch = 1
    gradient_accum = 8
    use_fp16 = False
    use_bf16 = False
    optim_choice = "adamw_torch"
    device_name = "CPU"

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=2,
    per_device_train_batch_size=per_device_batch,
    per_device_eval_batch_size=per_device_batch,
    gradient_accumulation_steps=gradient_accum,
    learning_rate=2e-4,
    warmup_steps=100,
    logging_steps=50,

    eval_strategy="steps",
    eval_steps=500,

    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    fp16=use_fp16,
    bf16=use_bf16,
    optim=optim_choice,

    gradient_checkpointing=True,
    max_grad_norm=1.0,
    lr_scheduler_type="cosine",

    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=2,
    group_by_length=True,
)

# ---------------------------
# Pretty print config
# ---------------------------
effective_bs = training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps
approx_steps_per_epoch = max(1, len(train_dataset_tokenized) // effective_bs)
approx_total_steps = approx_steps_per_epoch * int(training_args.num_train_epochs)

print("=" * 70)
print("TRAINING CONFIGURATION")
print("=" * 70)
print(f"  Device: {device_name}")
print(f"  Optimizer: {training_args.optim}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size per device: {training_args.per_device_train_batch_size}")
print(f"  Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"  Effective batch size: {effective_bs}")
print(f"  Total steps (approx): ~{approx_total_steps}")
print(f"  Precision: {'BF16' if use_bf16 else 'FP16' if use_fp16 else 'FP32'}")
print("=" * 70)


TRAINING CONFIGURATION
  Device: CPU
  Optimizer: OptimizerNames.ADAMW_TORCH
  Learning rate: 0.0002
  Epochs: 2
  Batch size per device: 1
  Gradient accumulation: 8
  Effective batch size: 8
  Total steps (approx): ~540
  Precision: FP32


In [29]:
import torch
from transformers import Trainer

# ---------------------------
# (Optional) clear CUDA cache
# ---------------------------
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    print(f"GPU memory before training (allocated): {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"GPU memory before training (reserved):  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

# ---------------------------
# Initialize Trainer
# ---------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_tokenized,
    eval_dataset=val_dataset_tokenized,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print("\n" + "=" * 70)
print("STARTING TRAINING")
print("=" * 70)
print(f"Training samples:   {len(train_dataset_tokenized):,}")
print(f"Validation samples: {len(val_dataset_tokenized):,}")
print("=" * 70 + "\n")

# ---------------------------
# Train with error handling
# ---------------------------
try:
    trainer.train()
    print("\n" + "=" * 70)
    print("✓ TRAINING COMPLETED SUCCESSFULLY!")
    if torch.cuda.is_available():
        print(f"✓ Peak GPU memory (allocated): {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
        print(f"✓ Current GPU memory (allocated): {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print("=" * 70)

except Exception as e:
    print("\n" + "=" * 70)
    print("❌ TRAINING FAILED")
    print("=" * 70)
    print(f"Error: {repr(e)}")
    if torch.cuda.is_available():
        print(f"GPU memory at failure (allocated): {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
        print(f"GPU memory at failure (reserved):  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")
    raise

# ---------------------------
# Save model + tokenizer
# ---------------------------
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"\n✓ Model + tokenizer saved to: {output_dir}")


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.



STARTING TRAINING
Training samples:   2,160
Validation samples: 240



huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
`use_cache=True` is incompat


❌ TRAINING FAILED
Error: RuntimeError('element 0 of tensors does not require grad and does not have a grad_fn')


RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [ ]:
# Training complete! Model is ready to use.
# Check the cells above for:
# - Saving the model
# - Testing inference
# - Downloading to your local machine

print("✓ Notebook execution complete!")
print("\nNext steps:")
print("  1. Save your trained model (see cell above)")
print("  2. Test inference with validation samples")
print("  3. Download model or save to Google Drive")
print("  4. Use the model for predictions on test set")

## 💾 Save and Download Trained Model

After training completes, you can save and download your model.

In [ ]:
# Save the final trained model and tokenizer
final_output_dir = "./qwen_telco_final"

print("Saving final model and tokenizer...")
model.save_pretrained(final_output_dir)
tokenizer.save_pretrained(final_output_dir)

print(f"✓ Model saved to: {final_output_dir}")
print("\nTo download the model from Colab:")
print("  1. Go to Files panel (left sidebar)")
print(f"  2. Navigate to {final_output_dir}")
print("  3. Right-click and select 'Download'")
print("\nOr use Google Drive to save it:")
print("  from google.colab import drive")
print("  drive.mount('/content/drive')")
print("  !cp -r ./qwen_telco_final /content/drive/MyDrive/")

## 🧪 Test Inference (Optional)

Test your trained model with a sample inference.

In [ ]:
# Test the trained model with a sample from validation set
import re

def extract_answer(text):
    """Extract the answer (C1-C8) from model response"""
    # Try boxed format first
    match = re.search(r'\\boxed\{(\d)\}', text)
    if match:
        num = int(match.group(1))
        return f"C{num}"
    
    # Try direct C1-C8 format
    match = re.search(r'\b(C[1-8])\b', text)
    if match:
        return match.group(1)
    
    return None

# Get a test sample from validation dataset (before tokenization)
test_idx = 2
test_question = val_dataset['question'][test_idx]
test_answer = val_dataset['answer'][test_idx]
test_id = val_dataset['id'][test_idx]

print("=" * 70)
print("TEST INFERENCE")
print("=" * 70)
print(f"Sample ID: {test_id}")
print(f"Expected answer: {test_answer}")
print(f"\nQuestion (first 800 chars):\n{test_question[:800]}...")

# Create chat messages for inference
inference_messages = [
    {
        "role": "system",
        "content": "You are a senior 5G RAN optimization engineer. Analyze the provided network data and identify the most likely root cause for throughput degradation. Provide your answer as the root cause label (C1-C8)."
    },
    {
        "role": "user",
        "content": test_question
    }
]

# Apply chat template
test_text = tokenizer.apply_chat_template(
    inference_messages,
    tokenize=False,
    add_generation_prompt=True
)
test_inputs = tokenizer(test_text, return_tensors="pt", truncation=True, max_length=2048)

# Move to device
if torch.cuda.is_available():
    test_inputs = {k: v.to("cuda") for k, v in test_inputs.items()}

# Generate
print("\n" + "=" * 70)
print("GENERATING PREDICTION...")
print("=" * 70)

with torch.no_grad():
    outputs = model.generate(
        **test_inputs,
        max_new_tokens=100,
        temperature=0.3,
        do_sample=True,
        top_p=0.9,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

# Decode response (only the generated part)
generated_ids = outputs[0][len(test_inputs["input_ids"][0]):]
response = tokenizer.decode(generated_ids, skip_special_tokens=True)

print("\nMODEL RESPONSE:")
print("=" * 70)
print(response)
print("=" * 70)

# Extract predicted answer
predicted = extract_answer(response)

print("\n" + "=" * 70)
print("RESULTS:")
print("=" * 70)
print(f"Expected answer: {test_answer}")
print(f"Predicted answer: {predicted}")

if predicted == test_answer:
    print("🎉 ✓ PREDICTION MATCHES EXPECTED ANSWER!")
else:
    print("❌ Prediction does not match expected answer")

print("=" * 70)


In [ ]:
# Show the full response for inspection
print("Full model response:")
print(response)


In [ ]:
# Test multiple samples to evaluate model performance
import re
from collections import defaultdict

def extract_answer(text):
    """Extract the answer (C1-C8) from model response"""
    # Try boxed format first
    match = re.search(r'\\boxed\{(\d)\}', text)
    if match:
        num = int(match.group(1))
        return f"C{num}"
    
    # Try direct C1-C8 format
    match = re.search(r'\b(C[1-8])\b', text)
    if match:
        return match.group(1)
    
    return None

print("=" * 70)
print("BATCH INFERENCE TEST (10 samples)")
print("=" * 70)

results = []
for i in range(min(10, len(val_dataset))):
    test_question = val_dataset['question'][i]
    test_answer = val_dataset['answer'][i]
    test_id = val_dataset['id'][i]
    
    # Create chat messages
    inference_messages = [
        {
            "role": "system",
            "content": "You are a senior 5G RAN optimization engineer. Analyze the provided network data and identify the most likely root cause for throughput degradation. Provide your answer as the root cause label (C1-C8)."
        },
        {
            "role": "user",
            "content": test_question
        }
    ]
    
    # Apply chat template and tokenize
    test_text = tokenizer.apply_chat_template(inference_messages, tokenize=False, add_generation_prompt=True)
    test_inputs = tokenizer(test_text, return_tensors="pt", truncation=True, max_length=2048)
    
    if torch.cuda.is_available():
        test_inputs = {k: v.to("cuda") for k, v in test_inputs.items()}
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **test_inputs,
            max_new_tokens=100,
            temperature=0.3,
            do_sample=False,  # Greedy for consistency
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    # Decode
    generated_ids = outputs[0][len(test_inputs["input_ids"][0]):]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True)
    
    # Extract prediction
    predicted = extract_answer(response)
    is_correct = (predicted == test_answer)
    
    results.append({
        "id": test_id,
        "expected": test_answer,
        "predicted": predicted,
        "correct": is_correct,
        "response": response
    })
    
    status = "✓" if is_correct else "❌"
    print(f"{i+1}. ID {test_id}: Expected={test_answer}, Predicted={predicted} {status}")

# Summary statistics
correct_count = sum(1 for r in results if r["correct"])
total_count = len(results)
accuracy = 100 * correct_count / total_count

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)
print(f"Correct: {correct_count}/{total_count}")
print(f"Accuracy: {accuracy:.1f}%")

# Show one example response
print("\n" + "=" * 70)
print("EXAMPLE RESPONSE (Sample 0):")
print("=" * 70)
print(results[0]["response"])
print("=" * 70)
